In [2]:
# Investigate label structure and feature schema
import pandas as pd
from pathlib import Path

DATA_ROOT = Path('/content/scratch/CICIoT2023_CSV/CSV')

# 1. Check README first
readme = DATA_ROOT.parent / 'README_CSV.pdf'
if readme.exists():
    print(f"README found at: {readme}")
    print(f"  Size: {readme.stat().st_size:,} bytes")
    print(f"  → Worth opening manually for the official schema; skipping PDF parse here.")
else:
    # check inside CSV/ too
    alt = DATA_ROOT / 'README_CSV.pdf'
    if alt.exists():
        print(f"README found at: {alt} ({alt.stat().st_size:,} bytes)")

# 2. Look at column schemas across 5 representative classes
print("\n=== Schema across representative classes ===")
samples = ['Benign_Final', 'DDoS-ICMP_Flood', 'Mirai-greip_flood',
           'Recon-PortScan', 'SqlInjection']
schemas = {}
for cls in samples:
    folder = DATA_ROOT / cls
    if not folder.exists(): continue
    f = next(folder.glob('*.csv'), None)
    if f is None: continue
    df = pd.read_csv(f, nrows=5)
    schemas[cls] = list(df.columns)
    print(f"\n  {cls} ({f.name}): {len(df.columns)} cols")
    print(f"    Last 5 cols: {list(df.columns[-5:])}")

# 3. Are schemas identical across classes?
print("\n=== Schema consistency check ===")
if schemas:
    base_cols = schemas[list(schemas.keys())[0]]
    all_same = all(cols == base_cols for cols in schemas.values())
    print(f"  All sampled classes have identical column order: {all_same}")
    if not all_same:
        for cls, cols in schemas.items():
            print(f"  {cls}: cols {len(cols)}")

# 4. Look at the actual first rows from Benign and one attack — do values look right?
print("\n=== Sample rows (Benign_Final, first 3) ===")
benign_file = next((DATA_ROOT / 'Benign_Final').glob('*.csv'))
df_b = pd.read_csv(benign_file, nrows=3)
print(df_b.to_string(max_cols=10))

print("\n=== Sample rows (DDoS-ICMP_Flood, first 3) ===")
ddos_file = next((DATA_ROOT / 'DDoS-ICMP_Flood').glob('*.csv'))
df_d = pd.read_csv(ddos_file, nrows=3)
print(df_d.to_string(max_cols=10))

# 5. Confirm a count of unique values in a column that might be implicit label
print("\n=== Quick row counts per class (first file only, for ordering of magnitude) ===")
for cls in samples:
    folder = DATA_ROOT / cls
    if not folder.exists(): continue
    f = next(folder.glob('*.csv'), None)
    if f is None: continue
    # fast row count without loading
    with open(f) as fp:
        n = sum(1 for _ in fp) - 1   # minus header
    print(f"  {cls} ({f.name}): {n:,} rows")

README found at: /content/scratch/CICIoT2023_CSV/CSV/README_CSV.pdf (69,751 bytes)

=== Schema across representative classes ===

  Benign_Final (BenignTraffic2.pcap.csv): 39 cols
    Last 5 cols: ['Std', 'Tot size', 'IAT', 'Number', 'Variance']

  DDoS-ICMP_Flood (DDoS-ICMP_Flood19.pcap.csv): 39 cols
    Last 5 cols: ['Std', 'Tot size', 'IAT', 'Number', 'Variance']

  Mirai-greip_flood (Mirai-greip_flood19.pcap.csv): 39 cols
    Last 5 cols: ['Std', 'Tot size', 'IAT', 'Number', 'Variance']

  Recon-PortScan (Recon-PortScan.pcap.csv): 39 cols
    Last 5 cols: ['Std', 'Tot size', 'IAT', 'Number', 'Variance']

  SqlInjection (SqlInjection.pcap.csv): 39 cols
    Last 5 cols: ['Std', 'Tot size', 'IAT', 'Number', 'Variance']

=== Schema consistency check ===
  All sampled classes have identical column order: True

=== Sample rows (Benign_Final, first 3) ===
   Header_Length  Protocol Type  Time_To_Live         Rate  fin_flag_number  ...          Std  Tot size       IAT  Number      Variance